# QC Processing Notebook
Development version that expects access to QC files via local filesystem

Breaks out the individual steps from the wrapper workflow.qc_process_and_model() so you can troubleshoot it step by step

In [ ]:
import os
import sys
from pathlib import Path
from earthscope_sfg_workflows.workflows.workflow_handler import WorkflowHandler
from earthscope_sfg_workflows.data_mgmt.model import GARPOSLayout
from earthscope_sfg_workflows.pipelines.config import ( SVPConfig,
    QCPipelineConfig, QCPinConfig, RinexConfig, PrideConfig, PositionUpdateConfig, KinConfig
)

from earthscope_sfg_workflows.data_mgmt.archives.earthscope_archive import EarthScopeArchive
from earthscope_sfg_tools.datamodels.metadata.earthscope.campaign import Campaign
import datetime


import logging
from earthscope_sfg_workflows.logging.loggers import set_all_logger_levels
logging.basicConfig(level=logging.INFO)
set_all_logger_levels(logging.INFO)


import pandas as pd
import matplotlib.pyplot as plt
import warnings
logging.getLogger("gnss_product_management").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning, module="earthscope_sfg_tools.tiledb_integration.arrays")

## Set these parameters

In [ ]:
# Main working directory for all data operations
main_dir = Path("/Users/mikegottlieb/data/sfg")

# Network, station, and campaign identifiers
NETWORK = "cascadia-gorda"
STATION = "NTH1"
CAMPAIGN = "2026_A_1126"
VESSEL = "1126"
CAMPAIGN_START = datetime.datetime(2026, 8, 18, tzinfo=datetime.timezone.utc)

# Directory containing raw QC .pin files to ingest
#raw_qc_data_dir = Path("/Users/mikegottlieb/data/PSN011153_all_data/NTH1_2025/20250812")

# Optional overrides to force rerunning individual steps even if they have been previously done
override_steps = {
    "generate_svp": True,
    "process_qc_pin": True,
    "build_rinex": True,
    "run_pride": True,
    "process_kinematic": True,
    "refine_shotdata": True
}

# =========================================================================
# Initialize Workflow
# =========================================================================
archive = EarthScopeArchive()
vessel = archive.load_vessel_metadata(VESSEL)  # real vessel metadata, independent of Campaign

campaign_stub = Campaign(
    name=CAMPAIGN,
    type="measure",          # or whatever this campaign's type is
    vesselCode=VESSEL,
    start=CAMPAIGN_START,
    end=datetime.datetime.now(datetime.timezone.utc),  # ongoing — today is fine
    vessel=vessel,
)



# Create the workflow handler
workflow = WorkflowHandler(directory=main_dir)

# Set the processing context (network/station/campaign)
workflow.set_network_station_campaign(
    network_id=NETWORK,
    station_id=STATION,
    campaign_id=CAMPAIGN,
)
workflow._session._campaign_meta = campaign_stub  

pipeline = workflow._session.pipeline           # ProcessingService for the active scope


cfg  = QCPipelineConfig(qcpin_config=QCPinConfig(override=override_steps["process_qc_pin"]),
                        rinex_config=RinexConfig(override=override_steps["build_rinex"], time_interval=24),
                        pride_config=PrideConfig(override=override_steps["run_pride"]),
                        kinematic_config=KinConfig(override=override_steps["process_kinematic"]),
                        position_update_config=PositionUpdateConfig(override=override_steps["refine_shotdata"]),
                        svp_config=SVPConfig(override=override_steps["generate_svp"]))

# Pre-processing Steps

In [ ]:

# =========================================================================
# Ingest QC Data (if not already done)
# =========================================================================

# Ingest raw QC files from the archive.  
# look for qc.zip first, then fall back to individual tar.gz files
# This step adds .pin files to the asset catalog
workflow.ingest_qc()    


In [ ]:
# =========================================================================
# Ingest CTD Data 
# =========================================================================

workflow.ingest_ctd_only(override=True)


In [ ]:
pipeline.run_qc('process_svp', config=cfg)

In [ ]:
pipeline.run_qc("process_qcpin",   config=cfg)  # pins -> shotdata + writes qc_gnss_obs.tdb

In [ ]:
pipeline.run_qc("build_rinex", config=cfg)  # qc_gnss_obs.tdb -> daily RINEX (the tdb2rnx step)

In [ ]:
pipeline.run_qc("run_pride", config=cfg)  # RINEX -> KIN + residuals (PRIDE-PPP)

In [ ]:
pipeline.run_qc("process_kinematic", config=cfg)  # KIN -> kinematic-position DataFrame 


In [ ]:
pipeline.run_qc("refine_shotdata", config=cfg)  # merge positions into final shotdata

### Now run GARPOS






In [ ]:
handler = workflow.modeling_get_garpos_handler()
qc = workflow._session.pipeline.get_qc()

# Surveys defined in the site metadata (or, if none are defined, a single
# survey spanning the full campaign data window).
available_surveys = handler.get_qc_surveys(qc.qcShotDataFinalTDB)
for i, s in enumerate(available_surveys):
    type_str = s.type.value if hasattr(s.type, "value") else s.type
    notes_str = f"  notes={s.notes}" if s.notes else ""
    number = i+1  
    print(f"[{number}] {s.id}  ({type_str})  {s.start} -> {s.end}{notes_str}")

In [ ]:
SURVEY_NUMBER = 1  # <-- pick from the list printed above, then re-run from here
survey = available_surveys[SURVEY_NUMBER-1]
print(f"Selected survey: {survey.id}")

In [ ]:
handler = workflow.modeling_get_garpos_handler()
qc = workflow._session.pipeline.get_qc()
gp = handler.parse_surveys_qc(
    override=False,
    shotdata_uri=qc.qcShotDataFinalTDB.uri,
    start=survey.start,
    end=survey.end,
    survey_id=survey.id,
)


In [ ]:
RUN_ID = "1"

GARPOS_CUSTOM_SETTINGS = {
    'maxloop': 20  # Maximum number of inversion loops
    }


In [ ]:

handler.run_garpos(surveys=gp, 
                   run_id=RUN_ID, 
                   iterations=2,
                   override=False, 
                   custom_settings=GARPOS_CUSTOM_SETTINGS)

In [ ]:
handler._plot_ts_results(
        survey_id=survey.id,
        run_id=RUN_ID,
        res_filter=10,
        savefig=True,
        showfig=False,
        results_dir=gp[0].results,
    )

In [ ]:
campaign = handler.station_session.ensure_campaign()
handler.current_garpos_survey_dir = GARPOSLayout.for_survey(campaign.qc / survey.id)
handler._plot_residuals_per_transponder_before_and_after(
            survey_id=survey.id, run_id=RUN_ID, savefig=True, showfig=True, point_size=10, subplots=False,
            # ymin=-2.5, ymax=1.5
        )


In [ ]:
handler._plot_remaining_residuals_per_transponder(
            survey_id=survey.id, run_id=RUN_ID, savefig=True, showfig=True, subplots=False, point_size=10,
        )

In [ ]:
pd.set_option('display.float_format', lambda x: '%.4f' % x)
gnatss_style_df = handler.to_gnatss_format_qc(gp, run_id=RUN_ID)
gnatss_style_df
handler.print_gnatss_format(gnatss_style_df, survey_id=survey.id)